In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, Dropout, Input, Flatten, LSTM, GRU, Conv1D, MaxPooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, f1_score, accuracy_score,
                             precision_score, recall_score, matthews_corrcoef,
                             confusion_matrix, roc_auc_score, average_precision_score)
from imblearn.over_sampling import RandomOverSampler
import time, warnings, os, random
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.utils.set_random_seed(seed)

set_seed(42)
print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [2]:
df = pd.read_csv(r"C:\Users\nuzha\IDD_project\IDD_DS\NF-ToN-IoT.csv")
df = df.fillna(0)
df.replace([np.inf, -np.inf], 0, inplace=True)

df['IN_BYTE_PER_PKT']    = df['IN_BYTES']  / (df['IN_PKTS']  + 1e-6)
df['OUT_BYTE_PER_PKT']   = df['OUT_BYTES'] / (df['OUT_PKTS'] + 1e-6)
df['TOTAL_BYTES']        = df['IN_BYTES']  + df['OUT_BYTES']
df['TOTAL_PKTS']         = df['IN_PKTS']   + df['OUT_PKTS']
df['FLOW_DUR_PER_PKT']   = df['FLOW_DURATION_MILLISECONDS'] / (df['TOTAL_PKTS'] + 1e-6)
df['IN_OUT_BYTES_RATIO'] = df['IN_BYTES'] / (df['OUT_BYTES'] + 1e-6)
df['IN_OUT_PKTS_RATIO']  = df['IN_PKTS']  / (df['OUT_PKTS']  + 1e-6)
df['TCP_SYN'] = df['TCP_FLAGS'].apply(lambda x: 1 if int(x) & 0x02 else 0)
df['TCP_ACK'] = df['TCP_FLAGS'].apply(lambda x: 1 if int(x) & 0x10 else 0)
df['TCP_FIN'] = df['TCP_FLAGS'].apply(lambda x: 1 if int(x) & 0x01 else 0)

le = LabelEncoder()
df['Label'] = le.fit_transform(df['Label'])
feature_cols = [c for c in df.columns if c not in ['IPV4_SRC_ADDR','IPV4_DST_ADDR','Label','Attack']]

zero_day_attacks = ['scanning','dos','backdoor','ransomware']
df = df.reset_index(drop=True)
zero_day_df = df[df['Attack'].str.lower().isin(zero_day_attacks)].copy()
normal_df   = df[~df['Attack'].str.lower().isin(zero_day_attacks)].copy()

cutoff1 = int(0.80 * len(normal_df))
dev_df      = normal_df.iloc[:cutoff1].copy()
test_normal = normal_df.iloc[cutoff1:].copy()
cutoff2 = int(0.90 * len(dev_df))
train_df = dev_df.iloc[:cutoff2].copy()
val_df   = dev_df.iloc[cutoff2:].copy()

def create_sequences(data, feature_cols, window_size=10, stride=5):
    sequences, labels, attacks = [], [], []
    feats = data[feature_cols].values.astype(np.float32)
    labs  = data['Label'].values
    atks  = data['Attack'].values
    for i in range(0, len(feats)-window_size+1, stride):
        sequences.append(feats[i:i+window_size])
        labels.append(labs[i+window_size-1])
        attacks.append(atks[i+window_size-1])
    return np.array(sequences,dtype=np.float32), np.array(labels), np.array(attacks)

WINDOW, STRIDE = 10, 5
X_train, y_train, _ = create_sequences(train_df, feature_cols, WINDOW, STRIDE)
X_val,   y_val,   _ = create_sequences(val_df,   feature_cols, WINDOW, STRIDE)
X_test_known, y_test_known, atk_known = create_sequences(test_normal, feature_cols, WINDOW, STRIDE)

scanning_df   = zero_day_df[zero_day_df['Attack'].str.lower()=='scanning']
dos_df        = zero_day_df[zero_day_df['Attack'].str.lower()=='dos']
backdoor_df   = zero_day_df[zero_day_df['Attack'].str.lower()=='backdoor']
ransomware_df = zero_day_df[zero_day_df['Attack'].str.lower()=='ransomware']

X_scan, y_scan, atk_scan = create_sequences(scanning_df, feature_cols, WINDOW, STRIDE)
X_dos,  y_dos,  atk_dos  = create_sequences(dos_df, feature_cols, WINDOW, STRIDE)
X_back, y_back, atk_back = create_sequences(backdoor_df, feature_cols, WINDOW, STRIDE)
X_rans, y_rans, atk_rans = create_sequences(ransomware_df, feature_cols, WINDOW, STRIDE)

X_test = np.concatenate([X_test_known, X_scan, X_dos, X_back, X_rans], axis=0)
y_test = np.concatenate([y_test_known, y_scan, y_dos, y_back, y_rans], axis=0)
attack_test = np.concatenate([atk_known, atk_scan, atk_dos, atk_back, atk_rans], axis=0)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_val   = scaler.transform(X_val.reshape(-1, X_val.shape[-1])).reshape(X_val.shape)
X_test  = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

n_s, t, f = X_train.shape
X_train_2d, y_train = RandomOverSampler(random_state=42).fit_resample(X_train.reshape(n_s,-1), y_train)
X_train = X_train_2d.reshape(-1, t, f)
n_features = X_train.shape[-1]
print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
print("n_features =", n_features)

Train: (316512, 10, 20) | Val: (21162, 10, 20) | Test: (64216, 10, 20)
n_features = 20


In [3]:
class StandardESN(Layer):
    def __init__(self, units=64, spectral_radius=0.9, leak_rate=0.3, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.spectral_radius = spectral_radius
        self.leak_rate = leak_rate
    def build(self, input_shape):
        feature_dim = int(input_shape[-1])
        self.W_in = self.add_weight(shape=(feature_dim, self.units), initializer='glorot_uniform', trainable=False)
        W = np.random.uniform(-1,1,(self.units,self.units)).astype(np.float32)
        W *= (np.random.rand(*W.shape) < 0.1).astype(np.float32)
        rho = np.max(np.abs(np.linalg.eigvals(W)))
        W *= (self.spectral_radius / (rho + 1e-8))
        self.W = self.add_weight(shape=(self.units,self.units), initializer=tf.constant_initializer(W), trainable=False)
        self.bias = self.add_weight(shape=(self.units,), initializer='zeros', trainable=False)
        super().build(input_shape)
    def call(self, inputs):
        batch = tf.shape(inputs)[0]
        state = tf.zeros((batch, self.units))
        def body(t, state):
            pre = tf.matmul(inputs[:,t,:], self.W_in) + tf.matmul(state, self.W) + self.bias
            return t+1, (1-self.leak_rate)*state + self.leak_rate*tf.tanh(pre)
        _, final = tf.while_loop(lambda t,s: t < tf.shape(inputs)[1], body, (0, state))
        return final

class HarmonicESN(Layer):
    def __init__(self, units=64, spectral_radius=0.9, leak_rate=0.3, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.spectral_radius = spectral_radius
        self.leak_rate = leak_rate
    def build(self, input_shape):
        feature_dim = int(input_shape[-1])
        self.W_in = self.add_weight(shape=(feature_dim, self.units), initializer='glorot_uniform', trainable=False)
        W = np.random.uniform(-1,1,(self.units,self.units)).astype(np.float32)
        W *= (np.random.rand(*W.shape) < 0.1).astype(np.float32)
        rho = np.max(np.abs(np.linalg.eigvals(W)))
        W *= (self.spectral_radius / (rho + 1e-8))
        self.W = self.add_weight(shape=(self.units,self.units), initializer=tf.constant_initializer(W), trainable=False)
        self.bias = self.add_weight(shape=(self.units,), initializer='zeros', trainable=False)
        super().build(input_shape)
    def call(self, inputs):
        batch = tf.shape(inputs)[0]
        state = tf.zeros((batch, self.units))
        def body(t, state):
            pre = tf.matmul(inputs[:,t,:], self.W_in) + tf.matmul(state, self.W) + self.bias
            return t+1, (1-self.leak_rate)*state + self.leak_rate*tf.sin(pre)
        _, final = tf.while_loop(lambda t,s: t < tf.shape(inputs)[1], body, (0, state))
        return final

class FB_HESN_Layer(Layer):
    def __init__(self, units=64, spectral_radius=0.9, leak_rate=0.3, omega_min=0.3, omega_max=3.5, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.spectral_radius = spectral_radius
        self.leak_rate = leak_rate
        self.omega_min = omega_min
        self.omega_max = omega_max
    def build(self, input_shape):
        feature_dim = int(input_shape[-1])
        self.W_in = self.add_weight(shape=(feature_dim, self.units), initializer='glorot_uniform', trainable=False)
        W = np.random.uniform(-1,1,(self.units,self.units)).astype(np.float32)
        W *= (np.random.rand(*W.shape) < 0.1).astype(np.float32)
        rho = np.max(np.abs(np.linalg.eigvals(W)))
        W *= (self.spectral_radius / (rho + 1e-8))
        self.W = self.add_weight(shape=(self.units,self.units), initializer=tf.constant_initializer(W), trainable=False)
        self.bias = self.add_weight(shape=(self.units,), initializer='zeros', trainable=False)
        omega = np.random.uniform(self.omega_min, self.omega_max, self.units).astype(np.float32)
        self.omega = self.add_weight(shape=(self.units,), initializer=tf.constant_initializer(omega), trainable=False)
        super().build(input_shape)
    def call(self, inputs):
        batch = tf.shape(inputs)[0]
        state = tf.zeros((batch, self.units))
        def body(t, state):
            pre = tf.matmul(inputs[:,t,:], self.W_in) + tf.matmul(state, self.W) + self.bias
            return t+1, (1-self.leak_rate)*state + self.leak_rate*tf.sin(self.omega * pre)
        _, final = tf.while_loop(lambda t,s: t < tf.shape(inputs)[1], body, (0, state))
        return final

def evaluate_and_store(y_true, y_pred, y_prob, attack_test, model, model_name, results_list, is_sklearn=False):
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, average='macro')
    mcc  = matthews_corrcoef(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)
    roc = roc_auc_score(y_true, y_prob)
    pr  = average_precision_score(y_true, y_prob)

    zd = {}
    for atk in ['scanning', 'dos', 'backdoor', 'ransomware']:
        mask = np.array([str(a).lower() == atk for a in attack_test])
        total = mask.sum()
        detected = (y_pred[mask] == 1).sum() if total > 0 else 0
        zd[atk] = (detected, total, detected/total if total > 0 else np.nan)

    zd_macro = np.nanmean([zd[a][2] for a in zd])
    zd_min   = np.nanmin([zd[a][2] for a in zd])

    # Parameter count
    if is_sklearn:
        total_params = 0
        trainable = 0
        infer_time = 0.0
    else:
        total_params = model.count_params()
        trainable = int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))
        # Inference time
        _ = model.predict(X_test[:128], verbose=0)
        t0 = time.time()
        _ = model.predict(X_test, batch_size=256, verbose=0)
        infer_time = time.time() - t0

    results_list.append({
        'Model': model_name,
        'Accuracy': round(acc, 4),
        'Macro-F1': round(f1, 4),
        'MCC': round(mcc, 4),
        'FPR': round(fpr, 4),
        'FNR': round(fnr, 4),
        'ROC-AUC': round(roc, 4),
        'PR-AUC': round(pr, 4),
        'Scanning': f"{zd['scanning'][0]}/{zd['scanning'][1]} ({zd['scanning'][2]:.4f})",
        'DoS': f"{zd['dos'][0]}/{zd['dos'][1]} ({zd['dos'][2]:.4f})",
        'Backdoor': f"{zd['backdoor'][0]}/{zd['backdoor'][1]} ({zd['backdoor'][2]:.4f})",
        'Ransomware': f"{zd['ransomware'][0]}/{zd['ransomware'][1]} ({zd['ransomware'][2]:.4f})",
        'ZD-MacroRecall': round(zd_macro, 4),
        'ZD-MinRecall': round(zd_min, 4),
        'Total Params': total_params,
        'Trainable Params': trainable,
        'Inference Time (s)': round(infer_time, 2)
    })
    print(f"{model_name} done | Macro-F1={f1:.4f} | ZD-Macro={zd_macro:.4f}")

In [4]:
results = []

# ----- Classical -----
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat  = X_test.reshape(X_test.shape[0], -1)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=20, n_jobs=-1, random_state=42, class_weight='balanced')
rf.fit(X_train_flat, y_train)
y_pred = rf.predict(X_test_flat)
y_prob = rf.predict_proba(X_test_flat)[:,1]
evaluate_and_store(y_test, y_pred, y_prob, attack_test, None, "Random Forest", results, is_sklearn=True)

# XGBoost
xgb = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, subsample=0.8,
                    colsample_bytree=0.8, eval_metric='logloss', random_state=42, n_jobs=-1)
xgb.fit(X_train_flat, y_train)
y_pred = xgb.predict(X_test_flat)
y_prob = xgb.predict_proba(X_test_flat)[:,1]
evaluate_and_store(y_test, y_pred, y_prob, attack_test, None, "XGBoost", results, is_sklearn=True)

# ----- Neural models (all units=64) -----
def train_neural(build_fn, name):
    tf.keras.backend.clear_session()
    set_seed(42)
    model = build_fn()
    model.compile(optimizer=Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(X_train, y_train, validation_data=(X_val,y_val), epochs=40, batch_size=256,
              callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)], verbose=0)
    y_prob = model.predict(X_test, batch_size=256, verbose=0)[:,1]
    y_pred = (y_prob >= 0.5).astype(int)
    evaluate_and_store(y_test, y_pred, y_prob, attack_test, model, name, results)

def build_dnn():
    inputs = Input(shape=(10, n_features))
    x = Flatten()(inputs)
    x = Dense(64, activation='relu')(x)      # ← start from 64
    x = Dropout(0.3)(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.2)(x)
    outputs = Dense(2, activation='softmax')(x)
    return Model(inputs, outputs)

def build_lstm():
    inputs = Input(shape=(10,n_features))
    x = LSTM(64)(inputs)
    x = Dropout(0.3)(x)
    x = Dense(32, activation='relu')(x)
    return Model(inputs, Dense(2, activation='softmax')(x))

def build_gru():
    inputs = Input(shape=(10,n_features))
    x = GRU(64)(inputs)
    x = Dropout(0.3)(x)
    x = Dense(32, activation='relu')(x)
    return Model(inputs, Dense(2, activation='softmax')(x))

def build_cnn_lstm():
    inputs = Input(shape=(10,n_features))
    x = Conv1D(64,3,activation='relu',padding='same')(inputs)
    x = MaxPooling1D(2)(x)
    x = LSTM(64)(x)
    x = Dropout(0.3)(x)
    x = Dense(32, activation='relu')(x)
    return Model(inputs, Dense(2, activation='softmax')(x))

def build_lstm_dnn():
    inputs = Input(shape=(10,n_features))
    x = LSTM(64)(inputs)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(32, activation='relu')(x)
    return Model(inputs, Dense(2, activation='softmax')(x))

def build_standard_esn():
    inputs = Input(shape=(10,n_features))
    x = StandardESN(units=64)(inputs)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.3)(x)
    return Model(inputs, Dense(2, activation='softmax')(x))

def build_harmonic_esn():
    inputs = Input(shape=(10,n_features))
    x = HarmonicESN(units=64)(inputs)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.3)(x)
    return Model(inputs, Dense(2, activation='softmax')(x))

def build_fb_hesn():
    inputs = Input(shape=(10,n_features))
    x = FB_HESN_Layer(units=64, spectral_radius=0.9, leak_rate=0.3, omega_min=0.3, omega_max=1.0)(inputs)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.3)(x)
    return Model(inputs, Dense(2, activation='softmax')(x))
    
def build_fb_hesn128():
    inputs = Input(shape=(10,n_features))
    x = FB_HESN_Layer(units=128, spectral_radius=0.9, leak_rate=0.3, omega_min=0.3, omega_max=1.0)(inputs)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.3)(x)
    return Model(inputs, Dense(2, activation='softmax')(x))

def build_fb_hesn_leaky():
    inputs = Input(shape=(10,n_features))
    x = FB_HESN_Layer(units=64, spectral_radius=0.9, leak_rate=0.1, omega_min=0.3, omega_max=1.0)(inputs)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.3)(x)
    return Model(inputs, Dense(2, activation='softmax')(x))

train_neural(build_dnn, "DNN")
train_neural(build_lstm, "LSTM-64")
train_neural(build_gru, "GRU-64")
train_neural(build_cnn_lstm, "CNN-LSTM-64")
train_neural(build_lstm_dnn, "LSTM-DNN-64")
train_neural(build_standard_esn, "Standard ESN-64")
train_neural(build_harmonic_esn, "Harmonic ESN-64")
train_neural(build_fb_hesn, "FB-HESN-64")
train_neural(build_fb_hesn128, "FB-HESN-128")
train_neural(build_fb_hesn_leaky, "FB-HESN-leaky")
# Master Table
df_master = pd.DataFrame(results)
print("\n" + "="*120)
print("MASTER COMPARISON TABLE")
print("="*120)
print(df_master.to_string(index=False))
df_master.to_csv("master_comparison_64.csv", index=False)
print("\nSaved to master_comparison_64.csv")

Random Forest done | Macro-F1=0.8948 | ZD-Macro=0.3670
XGBoost done | Macro-F1=0.9398 | ZD-Macro=0.5373

DNN done | Macro-F1=0.9793 | ZD-Macro=0.7068
LSTM-64 done | Macro-F1=0.9690 | ZD-Macro=0.6502
LSTM-DNN-64 done | Macro-F1=0.9507 | ZD-Macro=0.5906
Standard ESN-64 done | Macro-F1=0.9406 | ZD-Macro=0.6096
Harmonic ESN-64 done | Macro-F1=0.9330 | ZD-Macro=0.5636
FB-HESN-64 done | Macro-F1=0.9471 | ZD-Macro=0.6573
FB-HESN-128 done | Macro-F1=0.9712 | ZD-Macro=0.7253
FB-HESN-leaky done | Macro-F1=0.9109 | ZD-Macro=0.6341

MASTER COMPARISON TABLE
          Model  Accuracy  Macro-F1    MCC    FPR    FNR  ROC-AUC  PR-AUC           Scanning                DoS           Backdoor    Ransomware  ZD-MacroRecall  ZD-MinRecall  Total Params  Trainable Params  Inference Time (s)
  Random Forest    0.9151    0.8948 0.8092 0.0002 0.1114   0.9999  1.0000 4169/4292 (0.9713)  344/3542 (0.0971) 1377/3448 (0.3994) 0/27 (0.0000)          0.3670        0.0000             0                 0                

In [5]:
seeds = [42,43,44,45,46]
stab = {"Standard ESN-64":[], "Harmonic ESN-64":[], "FB-HESN-64":[]}

for seed in seeds:
    print(f"\n===== SEED {seed} =====")
    for name, builder in [("Standard ESN-64", build_standard_esn),
                          ("Harmonic ESN-64", build_harmonic_esn),
                          ("FB-HESN-64", build_fb_hesn)]:
        tf.keras.backend.clear_session()
        set_seed(seed)
        model = builder()
        model.compile(optimizer=Adam(0.001), loss='sparse_categorical_crossentropy')
        model.fit(X_train, y_train, validation_data=(X_val,y_val), epochs=40, batch_size=256,
                  callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)], verbose=0)
        y_prob = model.predict(X_test, batch_size=256, verbose=0)[:,1]
        y_pred = (y_prob>=0.5).astype(int)
        f1 = f1_score(y_test, y_pred, average='macro')
        mcc = matthews_corrcoef(y_test, y_pred)
        tn,fp,fn,tp = confusion_matrix(y_test,y_pred).ravel()
        fpr = fp/(fp+tn)
        zd = []
        for atk in ['scanning','dos','backdoor','ransomware']:
            mask = np.array([str(a).lower()==atk for a in attack_test])
            zd.append((y_pred[mask]==1).mean() if mask.sum()>0 else np.nan)
        stab[name].append([f1, mcc, fpr, np.nanmean(zd)] + zd)
        print(f"  {name}: F1={f1:.4f} | ZD={np.nanmean(zd):.4f}")

print("\n" + "="*90)
print("5-SEED STABILITY TABLE (mean ± std)")
print("="*90)
for name, runs in stab.items():
    runs = np.array(runs)
    m, s = np.nanmean(runs,0), np.nanstd(runs,0)
    print(f"\n{name}")
    print(f"  Macro-F1       : {m[0]:.4f} ± {s[0]:.4f}")
    print(f"  MCC            : {m[1]:.4f} ± {s[1]:.4f}")
    print(f"  FPR            : {m[2]:.4f} ± {s[2]:.4f}")
    print(f"  ZD-MacroRecall : {m[3]:.4f} ± {s[3]:.4f}")
    print(f"  Scanning       : {m[4]:.4f} ± {s[4]:.4f}")
    print(f"  DoS            : {m[5]:.4f} ± {s[5]:.4f}")
    print(f"  Backdoor       : {m[6]:.4f} ± {s[6]:.4f}")
    print(f"  Ransomware     : {m[7]:.4f} ± {s[7]:.4f}")


===== SEED 42 =====
  Harmonic ESN-64: F1=0.9754 | ZD=0.6913
  FB-HESN-64: F1=0.9850 | ZD=0.7463

===== SEED 45 =====
  Standard ESN-64: F1=0.9915 | ZD=0.7471
  Harmonic ESN-64: F1=0.9917 | ZD=0.7501
  FB-HESN-64: F1=0.9820 | ZD=0.7211

===== SEED 46 =====
  Standard ESN-64: F1=0.9814 | ZD=0.7295
  Harmonic ESN-64: F1=0.9899 | ZD=0.7674
  FB-HESN-64: F1=0.9833 | ZD=0.8034

5-SEED STABILITY TABLE (mean ± std)

Standard ESN-64
  Macro-F1       : 0.9686 ± 0.0186
  MCC            : 0.9395 ± 0.0348
  FPR            : 0.0013 ± 0.0009
  ZD-MacroRecall : 0.6837 ± 0.0520
  Scanning       : 0.9915 ± 0.0078
  DoS            : 0.8361 ± 0.2234
  Backdoor       : 0.8258 ± 0.1913
  Ransomware     : 0.0815 ± 0.0432

Harmonic ESN-64
  Macro-F1       : 0.9744 ± 0.0215
  MCC            : 0.9507 ± 0.0399
  FPR            : 0.0010 ± 0.0002
  ZD-MacroRecall : 0.7109 ± 0.0799
  Scanning       : 0.9937 ± 0.0048
  DoS            : 0.8743 ± 0.1913
  Backdoor       : 0.8792 ± 0.1163
  Ransomware     : 0.0963 ± 

In [6]:
print("Number of runs stored:")
for name, runs in stab.items():
    print(name, len(runs))

Number of runs stored:
Standard ESN-64 5
Harmonic ESN-64 5
FB-HESN-64 5


In [7]:
for name, runs in stab.items():
    print(f"\n{name}")
    for i, row in zip(seeds, runs):
        print(
            f"Seed {i}: "
            f"F1={row[0]:.4f}, "
            f"MCC={row[1]:.4f}, "
            f"FPR={row[2]:.4f}, "
            f"ZD={row[3]:.4f}, "
            f"Scan={row[4]:.4f}, "
            f"DoS={row[5]:.4f}, "
            f"Backdoor={row[6]:.4f}, "
            f"Ransomware={row[7]:.4f}"
        )


Standard ESN-64
Seed 42: F1=0.9406, MCC=0.8876, FPR=0.0007, ZD=0.6096, Scan=0.9765, DoS=0.3967, Backdoor=0.9171, Ransomware=0.1481
Seed 43: F1=0.9541, MCC=0.9121, FPR=0.0011, ZD=0.6409, Scan=0.9972, DoS=0.9763, Backdoor=0.4791, Ransomware=0.1111
Seed 44: F1=0.9752, MCC=0.9514, FPR=0.0029, ZD=0.6915, Scan=0.9972, DoS=0.9687, Backdoor=0.7631, Ransomware=0.0370
Seed 45: F1=0.9915, MCC=0.9832, FPR=0.0010, ZD=0.7471, Scan=0.9953, DoS=0.9709, Backdoor=0.9852, Ransomware=0.0370
Seed 46: F1=0.9814, MCC=0.9634, FPR=0.0005, ZD=0.7295, Scan=0.9914, DoS=0.8682, Backdoor=0.9843, Ransomware=0.0741

Harmonic ESN-64
Seed 42: F1=0.9330, MCC=0.8742, FPR=0.0010, ZD=0.5636, Scan=0.9842, DoS=0.4921, Backdoor=0.7039, Ransomware=0.0741
Seed 43: F1=0.9818, MCC=0.9642, FPR=0.0009, ZD=0.7821, Scan=0.9951, DoS=0.9692, Backdoor=0.9420, Ransomware=0.2222
Seed 44: F1=0.9754, MCC=0.9519, FPR=0.0014, ZD=0.6913, Scan=0.9963, DoS=0.9534, Backdoor=0.7784, Ransomware=0.0370
Seed 45: F1=0.9917, MCC=0.9835, FPR=0.0009, ZD

In [8]:
import numpy as np, pandas as pd, tensorflow as tf

def activation_for(layer):
    name = type(layer).__name__
    if name == 'StandardESN':  return np.tanh
    if name == 'HarmonicESN':  return np.sin
    om = layer.omega.numpy()                      # FB_HESN_Layer
    return lambda z: np.sin(om * z)

def esp_check(layer, X, K=100, n_pairs=20, eps=1e-3, seed=0):
    """Two copies, different random start states, same K consecutive input flows."""
    rng = np.random.default_rng(seed)
    Win, W, b = layer.W_in.numpy(), layer.W.numpy(), layer.bias.numpy()
    a, act = layer.leak_rate, activation_for(layer)
    U = X[: K // 10 + 1].reshape(-1, X.shape[-1])[:K]
    N = W.shape[0]
    worst_final, worst_step = 0.0, 0
    for _ in range(n_pairs):
        x1, x2 = rng.uniform(-1, 1, N), rng.uniform(-1, 1, N)
        first_ok = None
        for t, u in enumerate(U, 1):
            x1 = (1 - a) * x1 + a * act(u @ Win + x1 @ W + b)
            x2 = (1 - a) * x2 + a * act(u @ Win + x2 @ W + b)
            if first_ok is None and np.max(np.abs(x1 - x2)) < eps:
                first_ok = t
        worst_final = max(worst_final, np.max(np.abs(x1 - x2)))
        worst_step = max(worst_step, first_ok if first_ok else K + 1)
    return worst_final, worst_step

rows = []
for seed in [42, 43, 44, 45, 46]:
    for name, builder in [("Standard ESN-64", build_standard_esn),
                          ("Harmonic ESN-64", build_harmonic_esn),
                          ("FB-HESN-64", build_fb_hesn)]:
        tf.keras.backend.clear_session(); set_seed(seed)
        model = builder()
        layer = [l for l in model.layers if hasattr(l, 'W') and hasattr(l, 'W_in')][0]
        sig = np.linalg.norm(layer.W.numpy(), 2)                       # sigma_max(W)
        om_max = layer.omega.numpy().max() if hasattr(layer, 'omega') else 1.0
        d10, _  = esp_check(layer, X_test, K=10)
        d100, s = esp_check(layer, X_test, K=100)
        rows.append((seed, name, round(sig, 2), round(om_max * sig, 2), d10, d100, s))
print(pd.DataFrame(rows, columns=['seed','model','sigma_max','omega_max*sigma_max',
                                  'diff@K=10','diff@K=100','steps_to_eps']).to_string(index=False))

 seed           model  sigma_max  omega_max*sigma_max  diff@K=10   diff@K=100  steps_to_eps
   42 Standard ESN-64       1.71                 1.71   0.795855 3.799228e-06            57
   42 Harmonic ESN-64       1.71                 1.71   0.951704 3.260745e-05            68
   42      FB-HESN-64       1.71                 1.69   0.456310 1.058742e-08            42
   43 Standard ESN-64       1.97                 1.97   0.592001 7.499366e-07            56
   43 Harmonic ESN-64       1.97                 1.97   0.673115 1.707346e-05            67
   43      FB-HESN-64       1.97                 1.95   0.341293 1.075646e-08            42
   44 Standard ESN-64       1.46                 1.46   0.368372 1.642429e-08            41
   44 Harmonic ESN-64       1.46                 1.46   0.423219 2.397335e-07            46
   44      FB-HESN-64       1.46                 1.45   0.382443 3.327210e-10            35
   45 Standard ESN-64       1.45                 1.45   0.584099 2.174708e-05   